In [ ]:
#| default_exp helper.html

In [ ]:
#| export
import bs4
from bs4 import BeautifulSoup
import random
import re
from typing import NamedTuple, Optional, Union



from trouver.helper.regex import latex_indices, find_regex_in_text, replace_string_by_indices

In [ ]:
from fastcore.test import *
from nbdev.showdoc import show_doc

#### Consolidating special characters that are changed with the `__str__` function of `bs4.element.Tags` objects

In [ ]:
#| export
def html_tag_str(
        html_tag: bs4.element.Tag
        ) -> str:
    """
    Return the full string of `html_tag`, accounting for 
    special characters that `bs4` changes
    """
    special_chars = {'&lt;': '<', '&gt;': '>', '&amp;': '&'}
    text_to_return = str(html_tag)
    for special_char, replace_with in special_chars.items():
        text_to_return = text_to_return.replace(special_char, replace_with)
    return text_to_return

When using the `__str__` function of `bs4.element.Tag` objects, special characters such as `<`, `>` and `&` change into `&lt;`, `&gt;` and `&amp;`, etc. The `html_tag_str` function makes it so that these characters are changed back.

In [ ]:
soup = BeautifulSoup('', 'html.parser')
tag = soup.new_tag('span')
tag.string = '&hi<'
test_eq(html_tag_str(tag), '<span>&hi<</span>')

#### Handling less than `<` symbols in latex math mode strings

`BeautifulSoup`'s `html.parser` parses less than `<` symbols without a following space as the beginning of an HTML tag, even when the symbol `<` is used within a LaTeX math mode string. To get around this, we detect when this happens and add a space after these symbols.

In [ ]:
#| export
def find_lt_symbols_without_space_in_math_mode(
        text: str
        ) -> list[int]: # The index of  
    """
    Return the indices in `text` with math mode less than `<` symbols without
    a space that follows.
    """
    latex_inds = latex_indices(text)
    lt_pattern = re.compile(r'<(?! )')
    inds_of_lt_without_spaces_after = []
    for start, end in latex_inds:
        latex_str = text[start:end]
        relative_lt_inds = find_regex_in_text(latex_str, lt_pattern)
        inds_of_lt_without_spaces_after.extend([
            start + relative_lt_ind for relative_lt_ind, _ in relative_lt_inds
        ])
    return inds_of_lt_without_spaces_after 

In the following example, there are a few math mode strings with less than `<` symbols. Some of these symbols are followed by spaces and others are not.

In [ ]:
text = r"""
here is a math mode $a<b$. Here is another $a< b$.
Here is an in-line one:

$$ asdf <cbba$$

Here is another:

$$
asdf < basdf
$$
"""
output = find_lt_symbols_without_space_in_math_mode(text)
print(output)
test_eq(len(output), 2)
test_eq(text[output[0] + 1], 'b')
test_eq(text[output[1] + 1], 'c')

[23, 85]


In [ ]:
text_2 = r"""
<b>Now there is an HTML tag</b>. But it shouldn't be detected
because the tag is not within math mode text.
But this inequality is: $a <d$
"""
output = find_lt_symbols_without_space_in_math_mode(text_2)
print(output)
test_eq(len(output), 1)
test_eq(text_2[output[0] + 1], 'd')

[136]


In [ ]:
#| export
def add_space_to_lt_symbols_without_space(
        text: str
        ) -> str:
    """Add space after less than `<` symbols if the symbol
    is not followed by a space.
    """
    lt_wo_space_inds = find_lt_symbols_without_space_in_math_mode(text)
    lt_wo_space_ranges = [(ind, ind+1) for ind in lt_wo_space_inds]
    return replace_string_by_indices(
        text, replace_ranges=lt_wo_space_ranges,
        replace_with=['< '] * len(lt_wo_space_inds))
    

Let us again use `text` from the example for the `find_lt_symbols_without_space_in_math_mode` function:

In [ ]:
print(add_space_to_lt_symbols_without_space(text))
assert not find_lt_symbols_without_space_in_math_mode(add_space_to_lt_symbols_without_space(text))


here is a math mode $a< b$. Here is another $a< b$.
Here is an in-line one:

$$ asdf < cbba$$

Here is another:

$$
asdf < basdf
$$



The space-adding workaround above only protects `<` symbols that are inside detected math regions *and* not followed by a space, and it is applied by only some callers. The general fix escapes every `<` that cannot plausibly open an HTML tag **before** `bs4` parses: a `<` survives unescaped only when it matches a complete tag (quoted attribute values included) whose unquoted part contains no `$` or `\` — real HTML tags never contain LaTeX, while bs4-mangled garbage (e.g. `<b $$="" ...>`) always does. The escaped `&lt;` is decoded back to `<` by `bs4`, so the returned text keeps the original characters and positions.

In [ ]:
#| export
_HTML_TAG_OPEN_PATTERN = re.compile(
    r'</?[a-zA-Z][\w-]*(?:[^<>"\']|"[^"]*"|\'[^\']*\')*?/?>')

def _has_unquoted_latex(html_tag_str: str) -> bool:
    """Whether `html_tag_str` has a `$` or `\\` outside of quoted attribute values."""
    without_quotes = re.sub(r'"[^"]*"|\'[^\']*\'', '', html_tag_str)
    return '$' in without_quotes or '\\' in without_quotes

def escape_lt_symbols_that_do_not_open_html_tags(
        text: str, # The text in which to escape less than `<` symbols.
        ) -> str: # The same text with every `<` that cannot plausibly open an HTML tag replaced by `&lt;`.
    r"""Escape the `<` symbols in `text` that do not plausibly open an HTML tag.

    Every `<` inside a LaTeX math region is escaped (math never contains real
    HTML tags, and `$0<b>$` looks exactly like one). Outside math regions, a `<`
    is left as is only when it starts a complete tag (quoted attribute values
    may contain anything) whose unquoted part contains no LaTeX `$` or `\\`.
    This stops `bs4`'s `html.parser` from swallowing LaTeX such as
    `$a<b$ and $c>d$` as the attributes of a bogus tag. Regions are detected by
    `latex_indices`, so the fallback also covers dollar-sign pairing failures.
    """
    parts, pos = [], 0
    math_inds, math_i = latex_indices(text), 0
    while (ind := text.find('<', pos)) != -1:
        while math_i < len(math_inds) and math_inds[math_i][1] <= ind:
            math_i += 1
        in_math = math_i < len(math_inds) and math_inds[math_i][0] <= ind
        tag_match = None if in_math else _HTML_TAG_OPEN_PATTERN.match(text, ind)
        if tag_match is not None and not _has_unquoted_latex(tag_match.group()):
            parts.append(text[pos:tag_match.end()])
            pos = tag_match.end()
        else:
            parts.extend([text[pos:ind], '&lt;'])
            pos = ind + 1
    parts.append(text[pos:])
    return ''.join(parts)

In [ ]:
# Math `<` never looks like a real tag opening once scanned as such:
test_eq(
    escape_lt_symbols_that_do_not_open_html_tags(r'$a<b$ and $c>d$ tail'),
    r'$a&lt;b$ and $c>d$ tail')

# Real tags are kept verbatim, including `<` inside quoted attribute values:
test_eq(
    escape_lt_symbols_that_do_not_open_html_tags(r'<span typo="$a<b$">x</span> & $y<z$'),
    r'<span typo="$a<b$">x</span> & $y&lt;z$')

# bs4-mangled garbage (LaTeX in unquoted attribute positions) is escaped, not parsed:
assert escape_lt_symbols_that_do_not_open_html_tags(
    r'<b $$="" x="y\z">tail').startswith('&lt;b')

#### Removing HTML tags in a text and obtaining the data of the tags.

In [ ]:
markup = '<b>Hello</b>'
soup = BeautifulSoup(markup, 'html.parser')
tag = soup.b
new_str = soup.new_string(' World')
tag.append(new_str)

In [ ]:
new_str

' World'

In [ ]:
#| export
class HTMLTagWithIndices(NamedTuple):
    r"""
    A NamedTuple for encapsulating a single `bs4.element.Tag` object along
    with the range within a text that the tag encapsulates.
    """
    tag: bs4.element.Tag    
    start: int
    end: int

class StrAndHTMLTagsWithIndices(NamedTuple):
    """
    A NamedTuple for encapsulating a text (without HTML tags) along with 
    HTML tag data that is supposed to be in the text.
    """
    raw_text: str # The text without HTML tags
    tags: list[HTMLTagWithIndices]


In [ ]:
#| export
def remove_html_tags_in_text(
        text: str, # The text in which to remove the HTML tags.
        replace_with_attributes: Optional[Union[str, list[str]]] = None, # Attribute(s) within the HTML tags which should be used to replace the text of the tags. If `None`, then the texts are not replaced with the attributes. If multiple attributes are specified, then only one attribute is used to replace the text for each HTML tag (independently at random of other replacements). Each attribute's text has an equal chance of being selected for replacement. Repeats are ignored.
        definitely_replace: bool = False, # If `True` and if a given HTML tag has an attribute specified in `replace_with_attributes`, then the text for that tag will definitely be replaced by the text of one of the attributes. Otherwise, the original text and each attribute's text have an equal chance of being selected.
        seed: int = None # Random seed 
        # ) -> tuple[str, list[tuple[bs4.element.Tag, int, int]]]: # The text `removed` without HTML tags and a list whose elements consist of the removed HTML tags and the starting and ending indices of the text corresponding to the removed tags within `removed`.
        ) -> StrAndHTMLTagsWithIndices: # The text `removed` without HTML tags and a list whose elements consist of the removed HTML tags and the starting and ending indices of the text corresponding to the removed tags within `removed`.
    r"""Remove the HTML tags in `text`.

    HTML tags are assumed to be not nested.

    `<` symbols that cannot plausibly open a tag (LaTeX math, bs4-mangled
    garbage) are escaped before parsing so their text is never swallowed.

    """
    random.seed(seed)
    parsed_soup = BeautifulSoup(
        escape_lt_symbols_that_do_not_open_html_tags(text), 'html.parser')
    replace_with_attributes = _init_replace_with_attributes(
        replace_with_attributes)

    position = 0
    replaced_contents: list[HTMLTagWithIndices] = []
    for content in parsed_soup.contents:
        position = _process_content(
            parsed_soup, replace_with_attributes, definitely_replace, content,
            position, replaced_contents)
    text_to_return: str = html_tag_str(parsed_soup)
    return StrAndHTMLTagsWithIndices(text_to_return, replaced_contents)


def _init_replace_with_attributes(
        replace_with_attributes: Optional[Union[str, list[str]]]
        ) -> set[str]:
    if replace_with_attributes is None:
        replace_with_attributes = []
    elif isinstance(replace_with_attributes, str):
        replace_with_attributes = [replace_with_attributes]
    return set(replace_with_attributes)


# def _select_replacement_text(
#         content: bs4.element.Tag,
#         replace_with_attributes: set[str],
#         definitely_replace: bool) -> str:
#     if not replace_with_attributes:
#         return content.string
#     selection_pool = []    
#     if not definitely_replace:
#         selection_pool.append(content.string)
#     for attribute, value in content.attrs.items():
#         if attribute not in replace_with_attributes:
#             continue
#         selection_pool.append(value)
#     return random.choice(selection_pool)


def _select_replacement_text(
        content: bs4.element.Tag,
        replace_with_attributes: set[str],
        definitely_replace: bool) -> str:
    
    # Use .get_text() instead of .string to safely get inner content
    # content_text will be "Outer Inner" for <div>Outer <span>Inner</span></div>
    content_text = content.get_text() 

    if not replace_with_attributes:
        return content_text
        
    selection_pool = []    
    if not definitely_replace:
        selection_pool.append(content_text)
        
    for attribute, value in content.attrs.items():
        if attribute not in replace_with_attributes:
            continue
        selection_pool.append(value)
    
    # Handle the case where the pool might be empty (e.g. no string and no matching attrs)
    if not selection_pool: return ""
    
    selected = random.choice(selection_pool)
    return selected if selected is not None else ""

In [ ]:
#| export

def _process_content(
        parsed_soup: BeautifulSoup,
        replace_with_attributes: set[str],
        definitely_replace: bool,
        content,
        position: int,
        replaced_contents: list[HTMLTagWithIndices]
        ) -> int:
    
    if not isinstance(content, bs4.element.Tag):
        return position + len(content)
    replacement_text = _select_replacement_text(
        content, replace_with_attributes, definitely_replace)

    if replacement_text is None:
        replacement_text = ""
    
    try:
        replaced_content = content.replace_with(
            parsed_soup.new_string(replacement_text))
    except TypeError as e:
        raise e

    replaced_contents.append(HTMLTagWithIndices(
        replaced_content, position, position + len(replacement_text)))
        # (
        # replaced_content,
        # position,
        # position + len(replacement_text)))
    return position + len(replacement_text)
    

In [ ]:
#| hide
# 1. Test the NoneType fix (Empty tags)
# This confirms your change to handle None works
text_with_empty = "abstract<span></span>"
cleaned, _ = remove_html_tags_in_text(text_with_empty)
test_eq(cleaned, "abstract")

# 2. Test Content Preservation (The fix for your AssertionError)
# Using get_text() ensures the text inside tags isn't lost
text_with_content = "<b>Keep Me</b>"
cleaned_content, _ = remove_html_tags_in_text(text_with_content)
test_eq(cleaned_content, "Keep Me")

# 3. Test Attributes (The core feature of your function)
attr_text = '<span definition="formula">$$x+y$$</span>'
# Replace with the 'definition' attribute
cleaned_attr, _ = remove_html_tags_in_text(attr_text, replace_with_attributes='definition', definitely_replace=True)
test_eq(cleaned_attr, "formula")

# 4. Test nested-ish structure 
# Note: BS get_text() usually preserves the space if it's there
nested_text = "<div>Outer <span>Inner</span></div>"
cleaned_nested, _ = remove_html_tags_in_text(nested_text)
# If there's a space in the HTML, it should be in the result
test_eq(cleaned_nested, "Outer Inner")

In [ ]:
nested_text = "<div>Outer <span>Inner</span></div>"
cleaned_nested, _ = remove_html_tags_in_text(nested_text)

print(f"Result:   {repr(cleaned_nested)}")
print(f"Expected: {repr('Outer Inner')}")

# Check for specific hidden characters
if '\xa0' in cleaned_nested: print("Found Non-Breaking Space (\\xa0)!")
if '\n' in cleaned_nested: print("Found Newline (\\n)!")

Result:   'Outer Inner'
Expected: 'Outer Inner'


The `remove_html_tags_in_text` function removes HTML tags, preserving the underlying text by default.

In [ ]:
html = 'Let $K$ be a field. An <b definition="Abelian variety over a field">Abelian variety over $K$</b> is a variety that'
text_without_html_tags, removed_tags = remove_html_tags_in_text(html)
print(text_without_html_tags)

test_eq(text_without_html_tags, 'Let $K$ be a field. An Abelian variety over $K$ is a variety that')

Let $K$ be a field. An Abelian variety over $K$ is a variety that


In [ ]:
removed_tags[0][0].attrs

{'definition': 'Abelian variety over a field'}

In the following example, there is a less than `<` symbol, which is definitely not the opening of an HTML tag. The following verifies that the placeholder `&lt;` is not used to replace the less than symbol, which is what `bs4.BeautifulSoup`'s `html.parser` does.

In [ ]:
text = 'Hello, this has a less than symbol: $a< b$'
text, html_tags = remove_html_tags_in_text(text)
assert not html_tags
assert '< ' in text
assert 'lt' not in html_tags

The same applies to the greater than `>` symbol, and `&` symbols

In [ ]:
text = 'Hello, this has a greater than symbol: $a>b$'
text, html_tags = remove_html_tags_in_text(text)
assert not html_tags
assert '>' in text
assert 'gt' not in html_tags

text = r'Hello $$ f &= 3 \\ g &= 5'
text, html_tags = remove_html_tags_in_text(text)
assert not html_tags
assert '&' in text
assert '&amp;' not in html_tags

C:\Users\hyunj\AppData\Local\Temp\ipykernel_24224\1020901250.py:15: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  parsed_soup = BeautifulSoup(text, 'html.parser')


The next examples pin down the behavior that must **not change** when we harden `remove_html_tags_in_text` against LaTeX `<` symbols (characterization tests, per the coding standards' modify-with-a-test-net rule).

In [ ]:
html = 'Let $K$ be a field. An <b definition="Abelian variety over a field">Abelian variety over $K$</b> is a variety that'
text, tags = remove_html_tags_in_text(html)
test_eq(text, 'Let $K$ be a field. An Abelian variety over $K$ is a variety that')
test_eq(len(tags), 1)
test_eq(tags[0].tag.attrs['definition'], 'Abelian variety over a field')
test_eq(text[tags[0].start:tags[0].end], 'Abelian variety over $K$')

In [ ]:
# Quoted attribute values may legitimately contain `$`, `\` and even `<` (typo flow).
text, tags = remove_html_tags_in_text(r'<span typo="$a<b$">$\operatorname{Gal}(K)$</span>')
test_eq(text, r'$\operatorname{Gal}(K)$')
test_eq(tags[0].tag.attrs['typo'], r'$a<b$')

# Entities decode to their characters in the returned text; indices stay consistent.
text, tags = remove_html_tags_in_text('a &lt; b &amp; c <b definition="">d</b> e')
test_eq(text, 'a < b & c d e')
test_eq((tags[0].start, tags[0].end), (10, 11))

**Behavior change (was a defect).** `BeautifulSoup`'s `html.parser` used to turn
a math `<` into the start of a bogus tag and swallow every character up to the
next `>` — the swallowed text became *attributes* of the bogus tag and vanished
from the returned text. The asserts below record the fixed behavior; the lossy
pre-fix behavior is documented in the git history of this notebook.

In [ ]:
# With the escaping in `remove_html_tags_in_text`, math `<` survives verbatim and
# creates no tags.
text, tags = remove_html_tags_in_text(r'$a<b$ and $c>d$ tail')
test_eq(text, r'$a<b$ and $c>d$ tail')
assert not tags

# A real tag whose math content contains `<` keeps its closing tag, its attributes,
# and the exact span text.
text, tags = remove_html_tags_in_text('<span notation="">$x<y$</span> tail')
test_eq(text, '$x<y$ tail')
test_eq(len(tags), 1)
test_eq(tags[0].tag.attrs, {'notation': ''})
test_eq(tags[0].tag.get_text(), '$x<y$')
test_eq((tags[0].start, tags[0].end), (0, 5))

In [ ]:
# Bare tag lookalikes inside math are escaped too (`<b>` there is never markup):
test_eq(
    escape_lt_symbols_that_do_not_open_html_tags(r'for $0<b>$ plain tail'),
    r'for $0&lt;b>$ plain tail')
text, tags = remove_html_tags_in_text(r'for $0<b>$ some text</b> tail')
test_eq(text, r'for $0<b>$ some text</b> tail'.replace('</b>', ''))
assert not [t for t in tags if not (t.tag.has_attr('definition') or t.tag.has_attr('notation'))]

# A real mark whose math contains a bare lookalike keeps its nested marks:
text, tags = remove_html_tags_in_text(
    '<span notation="">$0<b>$</span> and <b definition="">def</b>')
test_eq(text, r'$0<b>$ and def')
test_eq([(t.name, s, e) for t, s, e in tags], [('span', 0, 6), ('b', 11, 14)])

The `remove_html_tags_in_text` function additionally returns a list with information about the tags that are removed. Each item in this list is a tuple `(tag, start, end)`, where `tag` is the tag that has been removed, and `start` and `end` are the indices within the string output `text_without_html_tags` of the function at which the text replacing the tag can be found.

In the example above (continued below), there is excactly one tag that is removed.

In [ ]:
print(removed_tags)
removed_tag, start, end = removed_tags[0]
print(text_without_html_tags[start:end])

test_eq(text_without_html_tags[start:end], 'Abelian variety over $K$')

[HTMLTagWithIndices(tag=<b definition="Abelian variety over a field">Abelian variety over $K$</b>, start=23, end=47)]
Abelian variety over $K$


In [ ]:
#| hide
html = '<span>Santa</span> want to eat some popcorn for once. He does not want <div>milk and cookies today</div>. \n And <a href="">this is a link.</a>'
text_without_html_tags, removed_tags = remove_html_tags_in_text(html)

test_eq(text_without_html_tags, 'Santa want to eat some popcorn for once. He does not want milk and cookies today. \n And this is a link.')
test_eq(len(removed_tags), 3)

removed_tag, start, end = removed_tags[0]
test_eq(text_without_html_tags[start:end], 'Santa')

removed_tag, start, end = removed_tags[1]
test_eq(text_without_html_tags[start:end], 'milk and cookies today')

removed_tag, start, end = removed_tags[2]
test_eq(text_without_html_tags[start:end], 'this is a link.')

The `remove_html_tags_in_text` function can be used to replace the underlying text of HTML tags with specified attribute values.

In the below example, the text has a tag which contains a `typo` attribute. Passing `'typo'` to the `replace_with_attributes` parameter and passing `True` to the `definitely_replace` parameter guarantees that the value of the `typo` attribute is used instead of the text of the tag.

In [ ]:
html = r'The following tag fixes a typo and simultaneously keeps around the data of that typo: <span typo="$\operatorname{Gul}(K)$">$\operatorname{Gal}(K)$</span>'
text_without_html_tags, removed_tags = remove_html_tags_in_text(html, replace_with_attributes='typo', definitely_replace=True)
print(text_without_html_tags)

test_eq(text_without_html_tags, 'The following tag fixes a typo and simultaneously keeps around the data of that typo: $\\operatorname{Gul}(K)$')

removed_tag, start, end = removed_tags[0]
test_eq(text_without_html_tags[start:end], '$\\operatorname{Gul}(K)$')

The following tag fixes a typo and simultaneously keeps around the data of that typo: $\operatorname{Gul}(K)$


If the `definitely_replace` parameter is `False` (which it is by default), then the original text might be preserved or it might be replaced.

In [ ]:
html = r'<span typo="$\operatorname{Gul}(K)$">$\operatorname{Gal}(K)$</span>'
possible_outputs = [
    r'$\operatorname{Gal}(K)$',
    r'$\operatorname{Gul}(K)$'
]
output, _ = remove_html_tags_in_text(html, replace_with_attributes='typo', definitely_replace=False)
assert output in possible_outputs

#### Adding HTML tag data

On the other hand, we may also need to add HTML tag data to a text.

In [ ]:
#| export
def add_HTML_tag_data_to_raw_text(
        text: str, # The text onto which to add HTML tags. This is assumed to contain no HTML tags.
        tags_and_locations: list[tuple[bs4.element.Tag, int, int]] # Each tuple consists of the tag object to add as well as the indices within `text` to. The ranges specified by the tuples are assumed to not overlap with one another.
        ) -> str: # The modification of `text` in which the tags are added at the specified locations; the ranges in `text` are replaced..
    """
    Add specified HTML tags to the specified locations/ranges in `text`.

    See the `add_HTML_tag_data_to_text` function for adding HTML
    tag data to text that may or may not already have HTML tags.
    """
    # sort by starting index
    tags_and_locations = sorted(
        tags_and_locations, key=lambda x: x[1])
    replace_ranges = [(start, end) for _, start, end in tags_and_locations]
    replace_with = [html_tag_str(html_tag) for html_tag, _, _ in tags_and_locations]
    return replace_string_by_indices(text, replace_ranges, replace_with)

In [ ]:
text = "Now this will have an HTML tag. This will also have an HTML tag too!"
tags_and_locations = [
    (BeautifulSoup('<span some_attr="hi">this</span>', 'html.parser'), 4,8),
    (BeautifulSoup('<div some_attr="hi">This</div>', 'html.parser'), 32,36)
]
output = add_HTML_tag_data_to_raw_text(text, tags_and_locations)
print(output)
test_eq(output, 'Now <span some_attr="hi">this</span> will have an HTML tag. <div some_attr="hi">This</div> will also have an HTML tag too!')

Now <span some_attr="hi">this</span> will have an HTML tag. <div some_attr="hi">This</div> will also have an HTML tag too!


Now let us look at the same example, with the order in `tags_and_locations` reversed.

In [ ]:
text = "Now this will have an HTML tag. This will also have an HTML tag too!"
tags_and_locations = [
    (BeautifulSoup('<div some_attr="hi">This</div>', 'html.parser'), 32,36),
    (BeautifulSoup('<span some_attr="hi">this</span>', 'html.parser'), 4,8)
]
output = add_HTML_tag_data_to_raw_text(text, tags_and_locations)
print(output)
test_eq(output, 'Now <span some_attr="hi">this</span> will have an HTML tag. <div some_attr="hi">This</div> will also have an HTML tag too!')

Now <span some_attr="hi">this</span> will have an HTML tag. <div some_attr="hi">This</div> will also have an HTML tag too!
